# Latent retokenization vs typo 2x2 figure

This notebook regenerates `paper/arXiv/figures/latent_retok_vs_typo_2x2_v3.png`.

- Retokenization pretraining curves are loaded from `figure_notebooks/latent_data/mu_retok_data.pkl` and `figure_notebooks/latent_data/var_retok_data.pkl`.
- Typo curves and post-training markers are loaded from the project digitization artifact because no native typo/post-training latent data is present in `latent_data`.


In [ ]:
from pathlib import Path
import os
import pickle
import re

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

ROOT = Path.cwd()
if ROOT.name == "figure_notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "figure_notebooks" / "latent_data"
DIGITIZED_DATA = ROOT / "outputs" / "figures" / "latent_retok_vs_typo_2x2_v3_digitized_data.md"
FIGURE_PATH = ROOT / "paper" / "arXiv" / "figures" / "latent_retok_vs_typo_2x2_v3.png"

LAYERS = [1, 8, 16, 24, 32]
POSTTRAIN_LABELS = ["Base", "SFT", "DPO", "Instruct"]
POSTTRAIN_X = np.array([1.45e6, 2.25e6, 3.45e6, 5.2e6])


In [ ]:
def load_pickle_curves(path):
    with path.open("rb") as f:
        raw = pickle.load(f)
    return {layer: (np.asarray(raw[layer][0]), np.asarray(raw[layer][1])) for layer in LAYERS}


def parse_digitized_tables(path):
    section = None
    tables = {}
    rows = []

    def flush():
        nonlocal rows
        if section and rows:
            tables[section] = rows
        rows = []

    for line in path.read_text().splitlines():
        if line.startswith("## "):
            flush()
            section = line.removeprefix("## ").strip()
            continue
        if not line.startswith("|") or line.startswith("| ---") or line.startswith("| x"):
            continue
        parts = [part.strip() for part in line.strip("|").split("|")]
        if len(parts) != 6:
            continue
        x_raw = parts[0].replace(",", "")
        x = x_raw if x_raw in POSTTRAIN_LABELS else float(x_raw)
        rows.append((x, {layer: float(value) for layer, value in zip(LAYERS, parts[1:])}))
    flush()
    return tables


def split_digitized_section(rows):
    pretraining = {layer: ([], []) for layer in LAYERS}
    posttraining = {label: {} for label in POSTTRAIN_LABELS}
    for x, values in rows:
        if isinstance(x, str):
            posttraining[x] = values
        else:
            for layer in LAYERS:
                pretraining[layer][0].append(x)
                pretraining[layer][1].append(values[layer])
    pretraining = {layer: (np.array(xs), np.array(ys)) for layer, (xs, ys) in pretraining.items()}
    return pretraining, posttraining


mu_retok = load_pickle_curves(DATA_DIR / "mu_retok_data.pkl")
var_retok = load_pickle_curves(DATA_DIR / "var_retok_data.pkl")
digitized = parse_digitized_tables(DIGITIZED_DATA)

_, mu_retok_post = split_digitized_section(digitized["Retokenization displacement mean"])
_, var_retok_post = split_digitized_section(digitized["Retokenization displacement variance"])
mu_typo, mu_typo_post = split_digitized_section(digitized["Typo displacement mean"])
var_typo, var_typo_post = split_digitized_section(digitized["Typo displacement variance"])


In [ ]:
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 22,
    "axes.labelsize": 20,
    "legend.fontsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "savefig.dpi": 160,
})

colors = plt.cm.viridis(np.linspace(0.14, 0.90, len(LAYERS)))
layer_colors = dict(zip(LAYERS, colors))


def plot_panel(ax, curves, posttraining, title, ylabel, ylim, dense=True):
    for layer in LAYERS:
        x, y = curves[layer]
        ax.plot(
            x,
            y,
            color=layer_colors[layer],
            marker="o",
            markersize=2.8 if dense else 5.5,
            linewidth=1.0,
            label=f"layer {layer}",
        )
        post_y = [posttraining[label][layer] for label in POSTTRAIN_LABELS]
        ax.plot(
            POSTTRAIN_X,
            post_y,
            color=layer_colors[layer],
            linestyle=":",
            linewidth=1.0,
            marker="*",
            markersize=13,
            markeredgecolor="black",
            markeredgewidth=0.6,
        )

    ax.set_xscale("log")
    ax.set_xlim(90, 6.3e6)
    ax.set_ylim(*ylim)
    ax.set_title(title, pad=6)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.28)
    ax.axvline(1e6, color="0.55", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.set_xticks([1e2, 1e3, 1e4, 1e5, 1e6])
    ax.legend(loc="upper right", frameon=True)


fig, axes = plt.subplots(2, 2, figsize=(13.0, 8.3), sharex=True)

plot_panel(axes[0, 0], mu_retok, mu_retok_post, "Retokenization displacement mean", r"$\mu(x)$", (-5, 300), dense=True)
plot_panel(axes[0, 1], var_retok, var_retok_post, "Retokenization displacement variance", r"$\mathrm{Var}(x)$", (-1, 17.5), dense=True)
plot_panel(axes[1, 0], mu_typo, mu_typo_post, "Typo displacement mean", r"$\mu(x)$", (-5, 300), dense=False)
plot_panel(axes[1, 1], var_typo, var_typo_post, "Typo displacement variance", r"$\mathrm{Var}(x)$", (-1, 17.5), dense=False)

axes[1, 0].set_xlabel("Pretraining step")
axes[1, 1].set_xlabel("Pretraining step")

for label, xpos in zip(POSTTRAIN_LABELS, POSTTRAIN_X):
    axes[1, 1].text(xpos, 1.65, label, rotation=90, ha="center", va="bottom", fontsize=11, color="0.25")

fig.tight_layout(pad=0.6)
# FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
# fig.savefig(FIGURE_PATH, bbox_inches="tight")
# FIGURE_PATH
